In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [2]:
data_gdynia_long_cat = pd.read_csv(
    "../../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long_cat.head()

,addr_street,function_clean,category
id,,,
219,Stefana Żeromskiego,administracja,organizacje publiczne
258,3 Maja,administracja,organizacje publiczne
374,Starowiejska,administracja,organizacje publiczne
495,10 Lutego,administracja,organizacje publiczne
86,Jana z Kolna,administracja,organizacje publiczne


In [3]:
data_gdynia_rank = data_gdynia_long_cat.groupby("addr_street", as_index=False)[
    "function_clean"
].agg("size")
data_gdynia_rank["rank"] = data_gdynia_rank["size"].rank(
    ascending=False, method="dense"
)
data_gdynia_rank.head(20)

,addr_street,size,rank
0,10 Lutego,54,5.0
1,3 Maja,29,9.0
2,Aleja Jana Pawła II,20,13.0
3,Aleja Marszałka Józefa Piłsudskiego,3,26.0
4,Antoniego Abrahama,120,2.0
5,Antoniego Hryniewickiego,26,11.0
6,Arkadiusza Rybickiego,12,19.0
7,Armii Krajowej,37,7.0
8,Dworcowa,6,24.0
9,Franciszka Żwirki i Stanisława Wigury,19,14.0


In [4]:
data_gdynia_shares_per_func = (
    data_gdynia_long_cat.groupby(["addr_street", "function_clean"])
    .agg(num_premises=("function_clean", "size"))
    .assign(
        num_total_premises=lambda d: d.groupby("addr_street")["num_premises"].transform(
            "sum"
        )
    )
    .assign(shares_per_premise=lambda d: d["num_premises"] / d["num_total_premises"])
)


In [5]:
data_gdynia_shares_per_func.head()

num_premises  num_total_premises  \
addr_street function_clean                                     
10 Lutego   administracja              1                  54   
            apteka                     1                  54   
            bank                       2                  54   
            bar                        1                  54   
            brak parteru               1                  54   

                            shares_per_premise  
addr_street function_clean                      
10 Lutego   administracja             0.018519  
            apteka                    0.018519  
            bank                      0.037037  
            bar                       0.018519  
            brak parteru              0.018519

In [6]:
assert np.isclose(
    (data_gdynia_shares_per_func.groupby("addr_street").sum())["shares_per_premise"], 1
).all()

In [10]:
data_gdynia_shares_per_func["rank"] = data_gdynia_shares_per_func.groupby(
    "addr_street"
)["num_premises"].rank(ascending=False, method="dense")
data_gdynia_shares_per_func = data_gdynia_shares_per_func.loc[
    data_gdynia_shares_per_func["rank"] <= 3, ["num_premises", "rank"]
].sort_values(["addr_street", "rank"], ascending=True)
data_gdynia_shares_per_func.head(50)

num_premises  \
addr_street                         function_clean                             
10 Lutego                           kebab                                  3   
                                    pustostan                              3   
                                    usługi zdrowie                         3   
                                    bank                                   2   
                                    centrum handlowe                       2   
                                    cukiernia                              2   
                                    handlowa ubrania                       2   
                                    jubiler                                2   
                                    kantor                                 2   
                                    kawiarnia                              2   
                                    piekarnia                              2   
                                    restauracja                            2   
                                    sklep spożywczy                        2   
                                    administracja                          1   
                                    apteka                                 1   
                                    bar                                    1   
                                    brak parteru                           1   
                                    delikatesy                             1   
                                    drogeria                               1   
                                    fryzjer                                1   
                                    gimnazjum                              1   
                                    handlowa                               1   
                                    handlowa antyki                        1   
                                    handlowa kolekcjonerska                1   
                                    handlowa medyczna                      1   
                                    handlowa używki                        1   
                                    hotel                                  1   
                                    informacja turystyczna                 1   
                                    mieszkaniowa wielorodzinna             1   
                                    pensjonat                              1   
                                    poczta                                 1   
                                    prokuratura                            1   
                                    przychodnia                            1   
                                    sklep erotyczny                        1   
                                    usługi energetyczne                    1   
                                    usługi pośrednictwo pracy              1   
                                    usługi szkolenia                       1   
                                    zegarmistrz                            1   
3 Maja                              mieszkaniowa wielorodzinna             4   
                                    restauracja                            3   
                                    centrum medyczne                       2   
                                    garaż                                  2   
                                    klub muzyczny                          2   
Aleja Jana Pawła II                 restauracja                            8   
                                    handlowa pamiątki                      3   
                                    szkoła żeglarska                       3   
                                    muzeum                                 2   
Aleja Marszałka Józefa Piłsudskiego mieszkaniowa wielorodzinna             1   
                                    restauracja                 

In [8]:
assert (data_gdynia_shares_per_func["rank"] <= 3).all()